# Lezione 10 · Il post-training

Il giudice che impara dalle coppie, le vele staccabili di LoRA, e la regola dell'esame.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

## Il giudice e le coppie

In [2]:
controlla("probabilità che la persona scelga la vincente", sigmoide(2 - 0), 0.88)

✓  probabilità che la persona scelga la vincente: calcolato 0.8808, nel libro 0.88


## LoRA: due tavole sottili

In [3]:
completa = 4096 * 4096
lora = 4096 * 8 + 8 * 4096
print(f"correzione completa: {completa/1e6:.1f} milioni di numeri; LoRA: {lora:,}")
controlla("frazione (%)", lora / completa * 100, 0.4, 0.02)
controlla("70 miliardi a 16 bit (GB)", 70e9 * 2 / 1e9, 140, 0)
controlla("70 miliardi a 4 bit (GB)", 70e9 * 0.5 / 1e9, 35, 0)

correzione completa: 16.8 milioni di numeri; LoRA: 65,536
✓  frazione (%): calcolato 0.3906, nel libro 0.4
✓  70 miliardi a 16 bit (GB): calcolato 140, nel libro 140
✓  70 miliardi a 4 bit (GB): calcolato 35, nel libro 35


## La regola dell'esame

Giusta +1, «non lo so» 0, sbagliata −k. Conviene rispondere solo se p > k / (1 + k).

In [4]:
for k, atteso in [(3, 0.75), (2, 2/3)]:
    controlla(f"soglia con errore −{k}", k / (1 + k), atteso, 0.001)

# il guadagno atteso, per chi vuole vederlo
for p in [0.5, 0.7, 0.8]:
    print(f"sicuro al {p:.0%}: rispondere vale {p - 3*(1-p):+.2f}, astenersi vale 0")

✓  soglia con errore −3: calcolato 0.75, nel libro 0.75
✓  soglia con errore −2: calcolato 0.6667, nel libro 0.666667
sicuro al 50%: rispondere vale -1.00, astenersi vale 0
sicuro al 70%: rispondere vale -0.20, astenersi vale 0
sicuro al 80%: rispondere vale +0.20, astenersi vale 0
